# ChromaDB CRUD Operations

CRUD means Create, Read, Update, and Delete. This notebook demonstrates adding records, inspecting them, semantic search, metadata filters, updates, upserts, and deletion. It uses a persistent local collection. Run in a disposable demo collection because deletion is permanent.


In [ ]:
# Install once if needed: %pip install chromadb
import chromadb
client = chromadb.PersistentClient(path="./chroma_data")
collection = client.get_or_create_collection(name="crud_demo")

# CREATE: use stable IDs; use upsert below for repeatable runs.
collection.upsert(
    ids=["doc1", "doc2", "doc3", "doc4", "doc5"],
    documents=[
        "Python is a popular programming language for AI.",
        "The weather today is sunny and warm.",
        "Machine learning models learn patterns from data.",
        "I enjoy playing football on weekends.",
        "Deep learning uses neural networks with many layers.",
    ],
    metadatas=[
        {"category": "tech"}, {"category": "weather"},
        {"category": "tech"}, {"category": "sports"},
        {"category": "tech"},
    ],
 )
print("Collection:", collection.name, "count:", collection.count())

# READ: preview and fetch by ID.
print("Preview:", collection.peek(limit=3))
record = collection.get(ids=["doc1", "doc3"], include=["documents", "metadatas"])
for doc_id, text, metadata in zip(record["ids"], record["documents"], record["metadatas"]):
    print(doc_id, metadata, text)

# Semantic query and category-filtered query.
query_result = collection.query(
    query_texts=["Tell me about AI and neural networks"],
    n_results=min(2, collection.count()),
    include=["documents", "metadatas", "distances"],
 )
for doc_id, text, distance in zip(query_result["ids"][0], query_result["documents"][0], query_result["distances"][0]):
    print(f"{doc_id} | distance={distance:.4f} | {text}")
filtered = collection.query(
    query_texts=["something fun to do outdoors"], n_results=min(3, collection.count()),
    where={"category": "sports"}, include=["documents", "metadatas", "distances"],
 )
print("Filtered IDs:", filtered["ids"][0])

# UPDATE: modify an existing ID.
collection.update(
    ids=["doc4"],
    documents=["I enjoy playing football and basketball on weekends."],
    metadatas=[{"category": "sports", "updated": True}],
 )
print("Updated record:", collection.get(ids=["doc4"], include=["documents", "metadatas"]))

# UPSERT: update doc5 and insert doc6.
collection.upsert(
    ids=["doc5", "doc6"],
    documents=["Deep learning uses neural networks with many layers and GPUs.", "The Eiffel Tower is located in Paris."],
    metadatas=[{"category": "tech"}, {"category": "travel"}],
 )
print("After upsert count:", collection.count())

# DELETE: remove a specific record, then demonstrate metadata deletion.
collection.delete(ids=["doc2"])
print("After deleting doc2:", collection.get()["ids"])
collection.delete(where={"category": "travel"})
print("After deleting travel records:", collection.get()["ids"])


## Summary

- `add`: insert new IDs; duplicate IDs are not intended for fresh inserts.
- `get`, `peek`, `count`: read and inspect records.
- `query`: nearest-neighbor semantic search, optionally filtered.
- `update`: modify existing records.
- `upsert`: insert or replace by ID; useful for repeatable ingestion.
- `delete`: permanently remove by IDs or metadata filter.

In a RAG index, preserve source/chunk metadata and delete obsolete chunks when a document changes. Check IDs and filters before destructive operations.